In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

SILVER_SCHEMA = f"{CATALOG}.{SCHEMA_PREFIX}_silver"
SILVER_TABLES = [f"{SILVER_SCHEMA}.customers", f"{SILVER_SCHEMA}.products"]

VACUUM_RETENTION_HOURS = 168

In [0]:
for table in SILVER_TABLES:
    spark.sql(f"OPTIMIZE {table}")
    spark.sql(f"VACUUM {table} RETAIN {VACUUM_RETENTION_HOURS} HOURS")

## Liquid Clustering vs ZORDER vs partitioning

| | Partitioning | ZORDER | Liquid Clustering |
|---|---|---|---|
| Changing the key | full rewrite | next `OPTIMIZE` | `ALTER TABLE ... CLUSTER BY`, no rewrite |
| Incremental | yes | no, rewrites a lot each time | yes |
| Main risk | many small files on high-cardinality columns | expensive repeated `OPTIMIZE` | needs a recent runtime |

Liquid Clustering is the current recommendation and is used here (`CLUSTER BY` in `00_setup`).